In [1]:
import mne
import os
import numpy as np
import pandas as pd
import scipy.io
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt

In [2]:
def batch_split(file_path, folder_name, train, specify_cluster = 12):
    # Load the .set file
    raw = mne.io.read_raw_eeglab(file_path)

    # Convert raw data to pandas DataFrame
    df = raw.to_data_frame()

    #removing "time" column
    df.drop(['time'], axis=1, inplace=True)

    # Filter the data
    fs = 512
    dt = 1/fs
    order = 2  # order of filter

    # Butterworth bandpass filter design
    b, a = butter(order, [fcutlow, fcuthigh], btype='band', fs=fs)

    # Apply the filter to the data
    df_filtered = filtfilt(b, a, df, axis=0, padtype='odd', padlen=3*(max(len(a), len(b))-1))

    #normalize the data
    max_values = df_filtered.max(axis=0)
    min_values = df_filtered.min(axis=0)
    row_mean = np.mean(df_filtered, axis=0)
    df_normalized = (df_filtered-row_mean)/(max_values-min_values)

    #converting to pandas

    #visualising the data
    #plt.plot(df_normalized_pd.iloc[:,0])

    df_normalized_pd = pd.DataFrame(df_normalized)

    #for a specific cluster
    if(specify_cluster == 2):
        df_normalized_pd = df_normalized_pd.iloc[:, [0, 1, 2, 4, 5, 10, 11, 12]]
    elif(specify_cluster == 5):
        df_normalized_pd = df_normalized_pd.iloc[:, [19, 20, 21, 28, 29, 30, 37, 38, 39]]
    elif(specify_cluster == 8):
        df_normalized_pd = df_normalized_pd.iloc[:, [46, 47, 48, 53, 54, 55, 57, 58, 59]]
    

    duration = 3 #batch duartion

    # Calculate the number of batches
    num_batches = len(df_normalized) // (fs * duration)

    # Calculate the maximum number of rows that can be evenly divided into batches
    max_rows = num_batches * fs * duration

    # Slice the DataFrame to the maximum number of rows
    df_normalized_sliced = df_normalized_pd.iloc[:max_rows]

    # Split the sliced data into batches
    batches = np.array_split(df_normalized_sliced, num_batches)

    train.extend(batches)    

    # os.makedirs(folder_name, exist_ok=True)

    # # Assuming 'batches' is the list of batches
    # for i, batch in enumerate(batches):
    #     batch_df = pd.DataFrame(batch)
    #     batch_df.to_csv(folder_name + f'/batch_{i}.csv', index=False)

### Choose cluster & freq


In [3]:
cluster = 12
band = 'delta'
#delta: 0.1-4
#theta: 4-7
#alpha 8-12
fcutlow = 0.1  # low cut frequency in Hz
fcuthigh = 4 # high cut frequency in Hz

### SCHIZOPHRENIC EC 

In [39]:
root = 'E:/sayan_24/GNN_aniket/data/RestingState_EC/Schizophrenia'
numbers = ["03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "13", 
           "14", "15", "16", "17", "18", "19", "20", "21", "22", "23", "24", "28"]
save_dir = 'E:/sayan_24/GNN_aniket/CSZ_select_' + band + '_cluster_' + str(cluster) + '/'
os.makedirs(save_dir, exist_ok = True)
case = 1

#splitting the data into train and test round robin fashion
for i in numbers:
    train_csz = []
    test = []
    for j in numbers:
        if i == j:
            folder_name = 'ms_08_00' + i + '_cleaned_EC'
            file_path = root + '/' + folder_name + '.set'
            batch_split(file_path, folder_name, test, specify_cluster=cluster)
        else:
            folder_name = 'ms_08_00' + i + '_cleaned_EC'
            file_path = root + '/' + folder_name + '.set'
            batch_split(file_path, folder_name, train_csz, specify_cluster=cluster)

    # Save train variable as .mat file
    save_path = os.path.join(save_dir, f'train_csz{case}.mat')
    scipy.io.savemat(save_path, {'train': train_csz})

    # Save test variable as .mat file
    save_path = os.path.join(save_dir, f'test{case}.mat')
    scipy.io.savemat(save_path, {'test': test})

    case += 1

C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.

In [7]:
root = 'E:/sayan_24/GNN_aniket_EO/data/RestingState_EO/HealthyControls'
numbers = ["02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "13", "14", "15", 
           "16", "17", "18", "19", "20", "21", "22", "23", "24", "25", "26", "27", "28", "29", 
           "30", "31", "32", "33", "34", "35", "36", "37", "38", "39", "58", "64"]
save_dir = 'E:/sayan_24/GNN_aniket_EO/CSZ_select_' + band + '_cluster_' + str(cluster) + '/'
os.makedirs(save_dir, exist_ok = True)
train_he = []
for i in numbers:
    folder_name = 'ms_00_00' + i + '_cleaned_E'
    file_path = root + '/' + folder_name + '.set'
    batch_split(file_path, folder_name, train_he, specify_cluster=cluster)

# Save train variable as .mat file
save_path = os.path.join(save_dir, 'train_he_full.mat')
scipy.io.savemat(save_path, {'train': train_he})

C:\Users\sundari\AppData\Local\Temp\ipykernel_15736\2789968522.py:3: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_15736\2789968522.py:3: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_15736\2789968522.py:3: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_15736\2789968522.py:3: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_15736\2789968522.py:3: Runtime

### HEALTHY EC 

In [41]:
root = 'E:/sayan_24/GNN_aniket/data/RestingState_EC/HealthyControls'
numbers = ["02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "13", "14", "15", 
           "16", "17", "18", "19", "20", "21", "22", "23", "24", "25", "26", "27", "28", "29", 
           "30", "31", "32", "33", "34", "35", "36", "37", "38", "39", "58", "64"]
save_dir = 'E:/sayan_24/GNN_aniket/HE_select_' + band + '_cluster_' + str(cluster) + '/'
os.makedirs(save_dir, exist_ok = True)
case = 1

#splitting the data into train and test round robin fashion
for i in numbers:
    train_he = []
    test = []
    for j in numbers:
        if i == j:
            folder_name = 'ms_00_00' + i + '_cleaned_EC'
            file_path = root + '/' + folder_name + '.set'
            batch_split(file_path, folder_name, test, specify_cluster=cluster)
        else:
            folder_name = 'ms_00_00' + i + '_cleaned_EC'
            file_path = root + '/' + folder_name + '.set'
            batch_split(file_path, folder_name, train_he, specify_cluster=cluster)

    # Save train variable as .mat file
    save_path = os.path.join(save_dir, f'train_he{case}.mat')
    scipy.io.savemat(save_path, {'train': train_he})

    # Save test variable as .mat file
    save_path = os.path.join(save_dir, f'test{case}.mat')
    scipy.io.savemat(save_path, {'test': test})

    case += 1

C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.

In [42]:
root = 'E:/sayan_24/GNN_aniket/data/RestingState_EC/Schizophrenia'
numbers = ["03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "13", 
           "14", "15", "16", "17", "18", "19", "20", "21", "22", "23", "24", "28"]
save_dir = 'E:/sayan_24/GNN_aniket/HE_select_' + band + '_cluster_' + str(cluster) + '/'
os.makedirs(save_dir, exist_ok = True)
train_csz = []
for i in numbers:
    folder_name = 'ms_08_00' + i + '_cleaned_EC'
    file_path = root + '/' + folder_name + '.set'
    batch_split(file_path, folder_name, train_csz, specify_cluster=cluster)

# Save train variable as .mat file
save_path = os.path.join(save_dir, 'train_csz_full.mat')
scipy.io.savemat(save_path, {'train': train_csz})

C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.py:3: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(file_path)
C:\Users\sundari\AppData\Local\Temp\ipykernel_10812\2761078875.